# Workshop Configuration

Set your **catalog** and **schema** below. All other notebooks pick up these values automatically.

Run notebooks in order: **01** → **02** → **03** → **03b** → **04** → … → **13**. Each notebook starts with `%run ./00_workshop_config` to load your settings.

## ✏️ Update these two values

In the cell below, change the **default values** on these two lines to match your environment:

```python
dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema",  "YOUR_SCHEMA",  "Schema")
```

For example, if your catalog is `acme_prod` and you want a schema called `genie_workshop`:

```python
dbutils.widgets.text("catalog", "acme_prod", "Catalog")
dbutils.widgets.text("schema",  "genie_workshop",  "Schema")
```

Everything else in this notebook can stay as-is.

In [ ]:
# ── Unity Catalog ─────────────────────────────────────────────
# Jobs pass catalog/schema via base_parameters (widgets). Interactive users set them above.
try:
    dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
    dbutils.widgets.text("schema",  "YOUR_SCHEMA",  "Schema")
except Exception:
    pass
try:
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA  = dbutils.widgets.get("schema")
except Exception:
    CATALOG = "YOUR_CATALOG"
    SCHEMA  = "YOUR_SCHEMA"
VOLUME  = "workshop_data"

## Genie Agent Settings

In [ ]:
# ── Genie Agents ─────────────────────────────────────────────
GENIE_SPACE_PREFIX  = "Manufacturing Quality Analytics"
SQL_WAREHOUSE_ID    = ""    # leave blank to auto-select a running warehouse

## Benchmark & Eval Settings

In [ ]:
# ── Benchmarks & Evals ───────────────────────────────────────
BENCHMARK_TOLERANCE_PCT = 0.01   # relative % for PASS — near-exact match (allows FP rounding)
BENCHMARK_WARN_PCT      = 1.0    # relative % for WARN
BENCHMARK_VERSION       = "v2"
USE_MLFLOW_EVALS        = True   # use mlflow.genai.evaluate() for scoring

## App Deployment

In [ ]:
# ── App ───────────────────────────────────────────────────────
APP_NAME = "manufacturing-genie"

## Monitoring & Alerting

In [ ]:
# ── Monitoring ────────────────────────────────────────────────
ALERT_ON_PASS_RATE_BELOW = 0.70  # alert if benchmark pass rate drops below this

## Derived Values (do not edit)

These are computed from the settings above. Used by all notebooks.

In [ ]:
# ── Derived (do not edit) ─────────────────────────────────────
FULL_SCHEMA = f"{CATALOG}.{SCHEMA}"

def full_table(name: str) -> str:
    """Return fully qualified 3-part table name."""
    return f"{CATALOG}.{SCHEMA}.{name}"

# Table names
PLANTS_TABLE             = full_table("plants")
PRODUCTION_LINES_TABLE   = full_table("production_lines")
OPERATORS_TABLE          = full_table("operators")
PRODUCTION_EVENTS_TABLE  = full_table("production_events")
QUALITY_METRICS_TABLE    = full_table("quality_metrics_daily")
SAFETY_INCIDENTS_TABLE   = full_table("safety_incidents")
EQUIPMENT_FEEDBACK_TABLE = full_table("equipment_feedback")
BENCHMARK_RUNS_TABLE     = full_table("genie_benchmark_runs")
CONFIG_TABLE             = full_table("workshop_config")

# Volume path
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}" if VOLUME else None

# Volume for workshop data
PREBUILD_VOLUME = "workshop_data"
PREBUILD_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{PREBUILD_VOLUME}/prebuild"

In [ ]:
# ── Backward compatibility alias ─────────────────────────────
# Many notebooks use `fqn` — this alias avoids updating every downstream cell
fqn = FULL_SCHEMA

In [ ]:
# ── Genie agent variants (built in 03 / 03b / 04) ────────────
# The workshop builds three agents on the SAME data to show that curation is
# what earns trust:
#   Baseline      (03)  — tables only, minimal instructions  ... the "before"
#   Metric View   (03b) — built on a governed metric view     ... semantic layer
#   Knowledge St. (04)  — measures/filters/fields/joins/etc.   ... the "after" (primary)
GENIE_TITLE_BASELINE    = f"{GENIE_SPACE_PREFIX} - Baseline"
GENIE_TITLE_CURATED     = f"{GENIE_SPACE_PREFIX} - Knowledge Store"
GENIE_TITLE_METRIC_VIEW = f"{GENIE_SPACE_PREFIX} - Metric View"

GENIE_DESC_BASELINE    = "Tables only, minimal instructions. The 'before' — baseline for comparison."
GENIE_DESC_CURATED     = "Fully curated Knowledge Store: measures, filters, fields, joins, synonyms, example SQL."
GENIE_DESC_METRIC_VIEW = "Built on a governed metric view (semantic layer) — deterministic KPIs."

# workshop_config keys, one per agent variant.
# CFG_KEY_CURATED is the PRIMARY agent that most downstream notebooks read.
CFG_KEY_CURATED     = "genie_space_id"
CFG_KEY_BASELINE    = "genie_space_id_blank"
CFG_KEY_METRIC_VIEW = "genie_space_id_metric_view"

# ── Metric views (created in 03b) ────────────────────────────
METRIC_VIEW_LINE_QUALITY = full_table("mv_line_quality")

# ── Idempotent upsert into the workshop_config table ─────────
def save_config_keys(new_rows):
    """Upsert rows into workshop_config (read-modify-overwrite).

    Each row is a dict with keys: key, value, space_name, space_url.
    Lets 03 / 03b / 04 each write their own agent id without clobbering the
    others, and makes re-running any of those notebooks safe.
    """
    from pyspark.sql import Row
    cols = ["key", "value", "space_name", "space_url"]
    merged = {}
    try:
        for r in spark.table(full_table("workshop_config")).collect():
            merged[r["key"]] = {c: r[c] for c in cols}
    except Exception:
        pass  # table does not exist yet — first writer creates it
    for nr in new_rows:
        merged[nr["key"]] = {c: nr.get(c) for c in cols}
    rows = [Row(**merged[k]) for k in sorted(merged)]
    (spark.createDataFrame(rows)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(full_table("workshop_config")))
    return merged

In [ ]:
# ── Print summary ────────────────────────────────────────
print("=" * 50)
print("Manufacturing Genie Workshop Config")
print("=" * 50)
print(f"Catalog         : {CATALOG}")
print(f"Schema          : {SCHEMA}")
print(f"Full schema     : {FULL_SCHEMA}")
print(f"Data path       : {PREBUILD_PATH}")
print(f"Warehouse ID    : {SQL_WAREHOUSE_ID or '(auto-select)'}")
print(f"Agent prefix    : {GENIE_SPACE_PREFIX}")
print(f"Benchmark ver   : {BENCHMARK_VERSION}")
print("=" * 50)